In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os, glob, zipfile

# Auto-search for any zip file in your Drive
zips = glob.glob("/content/drive/MyDrive/archive (2).zip", recursive=True)
print("Found zip files:")
for i, z in enumerate(zips):
    print(f"  [{i}] {z}  ({os.path.getsize(z)/1e6:.1f} MB)")

Found zip files:
  [0] /content/drive/MyDrive/archive (2).zip  (2504.9 MB)


In [ ]:
# If only one zip found, it picks automatically
# If multiple, change the index [0] to the correct one
zip_path = zips[0]
print(f"Extracting: {zip_path}")

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall("/content/face-mask-dataset")

print("✅ Done! Folder structure:")
for root, dirs, files_list in os.walk("/content/face-mask-dataset"):
    level = root.replace("/content/face-mask-dataset", "").count(os.sep)
    if level <= 3:
        indent = "  " * level
        print(f"{indent}{os.path.basename(root)}/")
        if level >= 2:
            print(f"{'  '*(level+1)}→ {len(files_list)} images")

Extracting: /content/drive/MyDrive/archive (2).zip
✅ Done! Folder structure:
face-mask-dataset/
  FMD_DATASET/
    without_mask/
      → 0 images
      complex/
        → 747 images
      simple/
        → 4000 images
    incorrect_mask/
      → 0 images
      mc/
        → 2500 images
      mmc/
        → 2500 images
    with_mask/
      → 0 images
      complex/
        → 789 images
      simple/
        → 4000 images


In [ ]:
import os, shutil
from pathlib import Path

src_base = "/content/face-mask-dataset/FMD_DATASET"
dst_base = "/content/dataset_flat"

# Map: class_name → list of subfolders to merge
class_map = {
    "with_mask"    : ["with_mask/simple",    "with_mask/complex"],
    "without_mask" : ["without_mask/simple", "without_mask/complex"],
    "incorrect_mask": ["incorrect_mask/mc",  "incorrect_mask/mmc"]
}

for class_name, subfolders in class_map.items():
    dst_class = os.path.join(dst_base, class_name)
    os.makedirs(dst_class, exist_ok=True)
    count = 0
    for sub in subfolders:
        src = os.path.join(src_base, sub)
        if not os.path.exists(src):
            print(f"  ⚠️ Not found: {src}")
            continue
        for fname in os.listdir(src):
            src_file = os.path.join(src, fname)
            # Rename to avoid conflicts between subfolders
            dst_file = os.path.join(dst_class, f"{sub.replace('/','_')}_{fname}")
            shutil.copy2(src_file, dst_file)
            count += 1
    print(f"✅ {class_name}: {count} images copied")

✅ with_mask: 4789 images copied
✅ without_mask: 4747 images copied
✅ incorrect_mask: 5000 images copied


In [ ]:
import os, shutil, random

src   = "/content/dataset_flat"
dst   = "/content/dataset_split"
SPLIT = {"train": 0.70, "val": 0.15, "test": 0.15}
random.seed(42)

for class_name in os.listdir(src):
    images = os.listdir(os.path.join(src, class_name))
    random.shuffle(images)

    total     = len(images)
    n_train   = int(total * SPLIT["train"])
    n_val     = int(total * SPLIT["val"])

    splits = {
        "train" : images[:n_train],
        "val"   : images[n_train:n_train+n_val],
        "test"  : images[n_train+n_val:]
    }

    for split_name, split_files in splits.items():
        out_dir = os.path.join(dst, split_name, class_name)
        os.makedirs(out_dir, exist_ok=True)
        for fname in split_files:
            shutil.copy2(
                os.path.join(src, class_name, fname),
                os.path.join(out_dir, fname)
            )

    print(f"✅ {class_name:15s} → "
          f"train:{len(splits['train'])}  "
          f"val:{len(splits['val'])}  "
          f"test:{len(splits['test'])}")

TRAIN_DIR = "/content/dataset_split/train"
VAL_DIR   = "/content/dataset_split/val"
TEST_DIR  = "/content/dataset_split/test"
print("\nPaths set!")

✅ without_mask    → train:3322  val:712  test:713
✅ incorrect_mask  → train:3500  val:750  test:750
✅ with_mask       → train:3352  val:718  test:719

Paths set!


In [ ]:
for split in ["train", "val", "test"]:
    print(f"\n{split.upper()}")
    for cls in os.listdir(f"/content/dataset_split/{split}"):
        n = len(os.listdir(f"/content/dataset_split/{split}/{cls}"))
        print(f"  {cls:20s} → {n} images")


TRAIN
  without_mask         → 3322 images
  incorrect_mask       → 3500 images
  with_mask            → 3352 images

VAL
  without_mask         → 712 images
  incorrect_mask       → 750 images
  with_mask            → 718 images

TEST
  without_mask         → 713 images
  incorrect_mask       → 750 images
  with_mask            → 719 images


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH    = 32

train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.15,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest"
)
val_gen  = ImageDataGenerator(rescale=1./255)
test_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE,
    batch_size=BATCH, class_mode="categorical"
)
val_data = val_gen.flow_from_directory(
    VAL_DIR, target_size=IMG_SIZE,
    batch_size=BATCH, class_mode="categorical", shuffle=False
)
test_data = test_gen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE,
    batch_size=BATCH, class_mode="categorical", shuffle=False
)

CLASS_NAMES = list(train_data.class_indices.keys())
NUM_CLASSES = len(CLASS_NAMES)
print("Classes:", train_data.class_indices)
# {'incorrect_mask': 0, 'with_mask': 1, 'without_mask': 2}

Found 10174 images belonging to 3 classes.
Found 2180 images belonging to 3 classes.
Found 2182 images belonging to 3 classes.
Classes: {'incorrect_mask': 0, 'with_mask': 1, 'without_mask': 2}


In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import AveragePooling2D, Dropout, Flatten, Dense, Input
from tensorflow.keras.models import Model

base_model = MobileNetV2(weights="imagenet", include_top=False,
                          input_tensor=Input(shape=(224, 224, 3)))
base_model.trainable = False

x = base_model.output
x = AveragePooling2D(pool_size=(7, 7))(x)
x = Flatten()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.3)(x)
output = Dense(NUM_CLASSES, activation="softmax")(x)  # 3 classes

model = Model(inputs=base_model.input, outputs=output)
print(f"Output layer: {NUM_CLASSES} classes → {CLASS_NAMES}")

/tmp/ipykernel_1541/2453673453.py:5: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(weights="imagenet", include_top=False,


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
Output layer: 3 classes → ['incorrect_mask', 'with_mask', 'without_mask']


In [ ]:
import tensorflow as tf
print("TF version :", tf.__version__)
print("GPU        :", tf.config.list_physical_devices('GPU'))

!pip install -q gradio opencv-python-headless scikit-learn

TF version : 2.19.0
GPU        : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    ModelCheckpoint("best_model.h5", monitor="val_accuracy",
                    save_best_only=True, verbose=1),
    EarlyStopping(monitor="val_loss", patience=5,
                  restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                      patience=3, verbose=1)
]

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    callbacks=callbacks
)
print("✅ Phase 1 done.")

Epoch 1/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 0s 605ms/step - accuracy: 0.7174 - loss: 0.6719

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(



Epoch 1: val_accuracy improved from None to 0.94220, saving model to best_model.h5



Epoch 1: finished saving model to best_model.h5
318/318 ━━━━━━━━━━━━━━━━━━━━ 257s 704ms/step - accuracy: 0.8210 - loss: 0.4603 - val_accuracy: 0.9422 - val_loss: 0.1798 - learning_rate: 1.0000e-04
Epoch 2/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 0s 536ms/step - accuracy: 0.9146 - loss: 0.2490
Epoch 2: val_accuracy improved from 0.94220 to 0.95505, saving model to best_model.h5



Epoch 2: finished saving model to best_model.h5
318/318 ━━━━━━━━━━━━━━━━━━━━ 183s 574ms/step - accuracy: 0.9203 - loss: 0.2271 - val_accuracy: 0.9550 - val_loss: 0.1439 - learning_rate: 1.0000e-04
Epoch 3/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 0s 531ms/step - accuracy: 0.9374 - loss: 0.1867
Epoch 3: val_accuracy improved from 0.95505 to 0.96101, saving model to best_model.h5


...


Epoch 4: finished saving model to best_model.h5
318/318 ━━━━━━━━━━━━━━━━━━━━ 180s 566ms/step - accuracy: 0.9468 - loss: 0.1564 - val_accuracy: 0.9628 - val_loss: 0.1109 - learning_rate: 1.0000e-04
Epoch 5/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 0s 532ms/step - accuracy: 0.9504 - loss: 0.1419
Epoch 5: val_accuracy did not improve from 0.96284
318/318 ━━━━━━━━━━━━━━━━━━━━ 182s 571ms/step - accuracy: 0.9512 - loss: 0.1418 - val_accuracy: 0.9628 - val_loss: 0.1116 - learning_rate: 1.0000e-04
Epoch 6/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 0s 526ms/step - accuracy: 0.9511 - loss: 0.1435
Epoch 6: val_accuracy improved from 0.96284 to 0.96606, saving model to best_model.h5



Epoch 6: finished saving model to best_model.h5
318/318 ━━━━━━━━━━━━━━━━━━━━ 180s 565ms/step - accuracy: 0.9547 - loss: 0.1346 - val_accuracy: 0.9661 - val_loss: 0.1013 - learning_rate: 1.0000e-04
Epoch 7/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 0s 534ms/step - accuracy: 0.9599 - loss: 0.1248
Epoch 7: val_accuracy improved from 0.96606 to 0.96927, saving model to best_model.h5



Epoch 7: finished saving model to best_model.h5
318/318 ━━━━━━━━━━━━━━━━━━━━ 182s 573ms/step - accuracy: 0.9587 - loss: 0.1216 - val_accuracy: 0.9693 - val_loss: 0.1025 - learning_rate: 1.0000e-04
Epoch 8/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 0s 523ms/step - accuracy: 0.9601 - loss: 0.1152
Epoch 8: val_accuracy did not improve from 0.96927
318/318 ━━━━━━━━━━━━━━━━━━━━ 179s 562ms/step - accuracy: 0.9623 - loss: 0.1127 - val_accuracy: 0.9674 - val_loss: 0.1003 - learning_rate: 1.0000e-04
Epoch 9/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 0s 528ms/step - accuracy: 0.9592 - loss: 0.1279
Epoch 9: val_accuracy did not improve from 0.96927
318/318 ━━━━━━━━━━━━━━━━━━━━ 180s 566ms/step - accuracy: 0.9602 - loss: 0.1200 - val_accuracy: 0.9679 - val_loss: 0.0958 - learning_rate: 1.0000e-04
Epoch 10/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 0s 520ms/step - accuracy: 0.9639 - loss: 0.1015
Epoch 10: val_accuracy improved from 0.96927 to 0.96972, saving model to best_model.h5



Epoch 10: finished saving model to best_model.h5
318/318 ━━━━━━━━━━━━━━━━━━━━ 178s 559ms/step - accuracy: 0.9636 - loss: 0.1032 - val_accuracy: 0.9697 - val_loss: 0.0909 - learning_rate: 1.0000e-04
Epoch 11/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 0s 539ms/step - accuracy: 0.9659 - loss: 0.1075
Epoch 11: val_accuracy did not improve from 0.96972
318/318 ━━━━━━━━━━━━━━━━━━━━ 183s 576ms/step - accuracy: 0.9642 - loss: 0.1064 - val_accuracy: 0.9651 - val_loss: 0.0956 - learning_rate: 1.0000e-04
Epoch 12/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 0s 525ms/step - accuracy: 0.9602 - loss: 0.1175
Epoch 12: val_accuracy improved from 0.96972 to 0.97156, saving model to best_model.h5



Epoch 12: finished saving model to best_model.h5
318/318 ━━━━━━━━━━━━━━━━━━━━ 198s 563ms/step - accuracy: 0.9645 - loss: 0.1057 - val_accuracy: 0.9716 - val_loss: 0.0909 - learning_rate: 1.0000e-04
Epoch 13/20
 25/318 ━━━━━━━━━━━━━━━━━━━━ 2:37 538ms/step - accuracy: 0.9677 - loss: 0.0884

In [ ]:
# Find the actual MobileNetV2 base model by name
from tensorflow.keras.optimizers import Adam

# Print all layers to find the base model
for i, layer in enumerate(model.layers):
    print(i, layer.name, type(layer).__name__)

In [ ]:
# Find base model (the one that has sub-layers)
base_model = None
for layer in model.layers:
    if hasattr(layer, 'layers') and len(layer.layers) > 10:
        base_model = layer
        print(f"Found base model: {layer.name}  ({len(layer.layers)} layers)")
        break

if base_model is None:
    print("Base model not found as sublayer — unfreezing directly")
    # Fallback: unfreeze last 30 layers of the whole model
    for layer in model.layers[-30:]:
        layer.trainable = True
else:
    for layer in base_model.layers[-30:]:
        layer.trainable = True

print(f"Total trainable layers: {sum(l.trainable for l in model.layers)}")

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_ft = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=callbacks
)
print("✅ Fine-tuning done.")

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model

model = load_model("best_model.h5")

loss, acc = model.evaluate(test_data)
print(f"\nTest Accuracy : {acc*100:.2f}%")
print(f"Test Loss     : {loss:.4f}")

y_pred = np.argmax(model.predict(test_data), axis=1)
y_true = test_data.classes

print("\n", classification_report(y_true, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title("Confusion Matrix")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

acc_all      = history.history["accuracy"]     + history_ft.history["accuracy"]
val_acc_all  = history.history["val_accuracy"] + history_ft.history["val_accuracy"]
loss_all     = history.history["loss"]         + history_ft.history["loss"]
val_loss_all = history.history["val_loss"]     + history_ft.history["val_loss"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(acc_all,     label="Train Acc")
ax1.plot(val_acc_all, label="Val Acc")
ax1.axvline(x=len(history.history["accuracy"])-1,
            color="gray", linestyle="--", label="Fine-tune start")
ax1.set_title("Accuracy")
ax1.legend()

ax2.plot(loss_all,     label="Train Loss")
ax2.plot(val_loss_all, label="Val Loss")
ax2.axvline(x=len(history.history["loss"])-1,
            color="gray", linestyle="--")
ax2.set_title("Loss")
ax2.legend()

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()

In [ ]:
model.save("mask_detector.h5")

from google.colab import files
files.download("mask_detector.h5")
print("✅ Model downloaded to your PC.")

In [ ]:
import gradio as gr
import numpy as np
import cv2
from tensorflow.keras.models import load_model

model = load_model("mask_detector.h5")

CLASS_NAMES = ['incorrect_mask', 'with_mask', 'without_mask']
IMG_SIZE = 224


def predict_mask(image):

    preview = image.copy()

    img = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    img = np.expand_dims(img, axis=0)

    preds = model.predict(img)[0]

    results = {CLASS_NAMES[i]: float(preds[i]) for i in range(len(CLASS_NAMES))}

    return preview, results


title = "😷 Face Mask Detection Dashboard"

description = """
Upload an image to detect:

✔ With Mask
❌ Without Mask
⚠ Incorrect Mask
"""


interface = gr.Interface(
    fn=predict_mask,
    inputs=gr.Image(type="numpy", label="Upload Image"),
    outputs=[
        gr.Image(label="Image Preview"),
        gr.Label(num_top_classes=3, label="Prediction")
    ],
    title=title,
    description=description
)

interface.launch(share=True)